# Measured Intrinsic Wavefront (MIW): OCS / CCS map reader

**Author:** Aaron Roodman &nbsp;|&nbsp; **Purpose:** read and plot the telescope-fixed (OCS) and
camera-fixed (CCS) Measured Intrinsic Wavefront maps for the Rubin AOS.

This is a standalone reader for the `intrinsic_split_maps.parquet` product. It needs only
`numpy`, `matplotlib`, `astropy`, and (for the optional lookup helper) `scipy` — **no LSST stack**
— so it can be used directly in the AOS online system.

### What the file contains
`intrinsic_split_maps.parquet` is an astropy Table on a regular focal-plane disk grid:

| column | meaning |
|---|---|
| `thx_deg`, `thy_deg` | field angle (degrees), OCS frame |
| `Z{j}_OCS` | telescope-fixed MIW for Noll Zernike *j* (µm) |
| `Z{j}_CCS` | camera-fixed MIW for Noll Zernike *j* (µm) |

Each measured map at rotator angle θ is modelled as `Z(r,φ) = O(r,φ) + C(r, φ - s·θ)`:
`O` is the telescope-fixed (OCS) part, `C` is the camera-fixed (CCS) part that rotates with the
Camera Rotator. For Zernikes fit OCS-only, `Z{j}_CCS` is identically zero. The table `.meta`
records the rotator angles used (`thetas_deg`), any `rotator_select` subset, the `ocs_only`
Zernikes, the rotation sign `s`, and `build_from`.

## 1. Parameters

In [ ]:
from pathlib import Path
import re
import numpy as np
import matplotlib.pyplot as plt
from astropy.table import Table

# Versioned OCS/CCS maps from the calibration store (see aos/calibration/).
# The version label is in the filename, so it travels with the file when deployed.
version = "v1"
maps_path = f"calibration/miw/intrinsic_split_maps_{version}.parquet"
# For online/AOS use, set maps_path to an absolute path to the deployed file.

zernikes = "all"        # "all", or a list of Noll indices e.g. [4, 5, 6, 7, 8]
fp_radius_deg = 1.75    # focal-plane radius for plot limits (deg)
pct = 98                # color scale percentile (per Zernike)

## 2. Read the maps

In [ ]:
t = Table.read(maps_path, format="parquet")
thx = np.asarray(t["thx_deg"], dtype=float)
thy = np.asarray(t["thy_deg"], dtype=float)

# discover the Zernikes present from the Z{j}_OCS columns
js = sorted({int(m.group(1)) for c in t.colnames
             for m in [re.match(r"Z(\d+)_OCS$", c)] if m})
if zernikes != "all":
    js = [j for j in js if j in zernikes]
ocs_only = set(int(j) for j in (t.meta.get("ocs_only") or []))

print(f"{len(thx)} field points;  Zernikes available: {js}")
print("metadata:")
for k, v in t.meta.items():
    print(f"  {k}: {v}")

## 3. Helper: focal-plane map

In [ ]:
def plot_map(ax, vals, title, vlim, cmap="RdBu_r"):
    """Filled-contour map of a per-field-point value on (thx, thy)."""
    v = np.asarray(vals, dtype=float)
    fin = np.isfinite(v)
    tcf = ax.tricontourf(thx[fin], thy[fin], v[fin],
                         levels=np.linspace(-vlim, vlim, 21), cmap=cmap, extend="both")
    ax.add_patch(plt.Circle((0, 0), fp_radius_deg, fill=False, ec="k", lw=0.6, alpha=0.4))
    ax.set_aspect("equal")
    ax.set_xlim(-fp_radius_deg, fp_radius_deg); ax.set_ylim(-fp_radius_deg, fp_radius_deg)
    ax.set_title(title, fontsize=9)
    ax.set_xlabel("thx [deg]"); ax.set_ylabel("thy [deg]")
    return tcf


def vlim_for(*arrays):
    """Symmetric color limit from the robust percentile over the given arrays."""
    vals = np.concatenate([np.asarray(a, float)[np.isfinite(a)] for a in arrays])
    return max(float(np.nanpercentile(np.abs(vals), pct)), 1e-6) if vals.size else 1.0

## 4. Plot the OCS and CCS maps
One row per Zernike: telescope-fixed (OCS) on the left, camera-fixed (CCS) on the right, shared color scale.

In [ ]:
for j in js:
    O = np.asarray(t[f"Z{j}_OCS"], dtype=float)
    C = np.asarray(t[f"Z{j}_CCS"], dtype=float)
    vlim = vlim_for(O, C)
    fig, axs = plt.subplots(1, 2, figsize=(11, 4.6), layout="constrained")
    tcf = plot_map(axs[0], O, f"Z{j}  OCS (telescope)", vlim)
    ccs_note = "   [OCS-only: C≈0]" if j in ocs_only else ""
    plot_map(axs[1], C, f"Z{j}  CCS (camera){ccs_note}", vlim)
    fig.colorbar(tcf, ax=axs, shrink=0.85, label="µm")
    fig.suptitle(f"Measured Intrinsic Wavefront — Z{j}", fontsize=12)
    plt.show()

## 5. Reference values for the AOS online system

For a fixed field point at radius `r_query_deg` (here 1.7°) this tabulates, for **every** Zernike,
its **OCS** (telescope-fixed) part, its **CCS** (camera-fixed) part, and the **total** measured
OCS-frame wavefront at several **rotator angles**. It is a check fixture: the online code that
consumes these maps should reproduce the `total@θ` column after applying its own CCS→OCS rotation.

The model matches the pipeline exactly (`s = +1`, the OCS→CCS = R(−θ) convention):

$$Z_j(\theta) \;=\; O_j \;+\; e^{\,i\,\mathrm{spin}\,s\,\theta}\;\; C_j\big(r,\;\varphi - s\theta\big)$$

where `O_j = Z{j}_OCS`, `C_j = Z{j}_CCS`, and a (cos, sin) doublet forms the **complex** field
`Z_cos + i·Z_sin` (cos → real part, sin → imaginary part). Two things happen as the rotator turns:
the camera-fixed field is sampled at the rotated azimuth `φ − sθ`, **and** the phase
`exp(i·spin·s·θ)` mixes the cos/sin members — that mixing *is* the CCS→OCS conversion to verify.
Scalar (m=0) terms have spin 0 (no cos/sin mixing, but the camera field still rotates in azimuth);
OCS-only terms have `C ≈ 0`, so their total is rotator-independent — a useful negative control.

In [ ]:
from scipy.interpolate import LinearNDInterpolator

# ---- field point and rotator angles to evaluate (edit freely) ----
r_query_deg = 1.7                       # focal-plane radius (deg)
az_query_deg = 30.0                     # OCS azimuth of the field point (deg)
rotator_angles_deg = [-20.0, 0.0, 20.0]

# ---- conventions (must match the AOS pipeline; see the markdown above) ----
s = int(t.meta.get("rotation_sign", 1))   # field-rotation sense; OCS->CCS = R(-theta)
spin_sign = 1                              # doublet phase sign (pipeline default +1)

# Noll index -> (radial n, signed azimuthal m); spin = |m|, cos has m>0, sin m<0.
NOLL_NM = {4: (2, 0), 5: (2, -2), 6: (2, 2), 7: (3, -1), 8: (3, 1), 9: (3, -3),
           10: (3, 3), 11: (4, 0), 12: (4, 2), 13: (4, -2), 14: (4, 4), 15: (4, -4),
           16: (5, 1), 17: (5, -1), 18: (5, 3), 19: (5, -3), 20: (5, 5), 21: (5, -5),
           22: (6, 0), 23: (6, -2), 24: (6, 2), 25: (6, -4), 26: (6, 4)}


def miw_interpolator(j, frame="OCS"):
    """LinearNDInterpolator for Z{j}_{frame}: call f(thx_deg, thy_deg) -> value (µm)."""
    v = np.asarray(t[f"Z{j}_{frame}"], dtype=float)
    fin = np.isfinite(v)
    return LinearNDInterpolator(np.column_stack([thx[fin], thy[fin]]), v[fin])


_OCS = {j: miw_interpolator(j, "OCS") for j in js}
_CCS = {j: miw_interpolator(j, "CCS") for j in js}


def _v(interp, xq, yq):
    return float(np.asarray(interp(xq, yq)))


def reconstruct(thx_q, thy_q, theta_deg):
    """Measured OCS-frame Zernikes at field point (thx_q, thy_q), rotator theta_deg.

    Returns {j: (ocs, ccs_raw, ccs_at_theta, total)} in µm, where
        ocs          = O_j               (telescope-fixed; rotator-independent)
        ccs_raw      = C_j at the point   (camera-fixed value, theta=0 azimuth)
        ccs_at_theta = exp(i*spin*s*th) * C_j(r, phi - s*th)  converted to the OCS frame
        total        = ocs + ccs_at_theta (what the online code should reproduce)
    """
    th = np.deg2rad(theta_deg)
    r = np.hypot(thx_q, thy_q)
    phi = np.arctan2(thy_q, thx_q)
    xc, yc = r * np.cos(phi - s * th), r * np.sin(phi - s * th)   # CCS azimuth phi - s*th
    out, done = {}, set()
    for j in js:
        if j in done:
            continue
        n, m = NOLL_NM[j]
        if m == 0:                                  # scalar: spin 0, no cos/sin mixing
            O = _v(_OCS[j], thx_q, thy_q)
            craw = _v(_CCS[j], thx_q, thy_q)
            cth = _v(_CCS[j], xc, yc)               # camera field at rotated azimuth
            out[j] = (O, craw, cth, O + cth)
            done.add(j)
            continue
        k = next((p for p in js if NOLL_NM.get(p) == (n, -m)), None)
        if k is None:                               # unpaired term: treat like its own field
            O = _v(_OCS[j], thx_q, thy_q)
            craw = _v(_CCS[j], thx_q, thy_q)
            cth = np.exp(1j * spin_sign * abs(m) * s * th) * _v(_CCS[j], xc, yc)
            out[j] = (O, craw, cth.real, O + cth.real)
            done.add(j)
            continue
        jc, jsn = (j, k) if m > 0 else (k, j)       # cos member has m>0
        spin = abs(m)
        Ocplx = _v(_OCS[jc], thx_q, thy_q) + 1j * _v(_OCS[jsn], thx_q, thy_q)
        Craw = _v(_CCS[jc], thx_q, thy_q) + 1j * _v(_CCS[jsn], thx_q, thy_q)
        Cth = np.exp(1j * spin_sign * spin * s * th) * \
            (_v(_CCS[jc], xc, yc) + 1j * _v(_CCS[jsn], xc, yc))
        tot = Ocplx + Cth
        out[jc] = (Ocplx.real, Craw.real, Cth.real, tot.real)
        out[jsn] = (Ocplx.imag, Craw.imag, Cth.imag, tot.imag)
        done.update((jc, jsn))
    return out


# ---- build and print the reference table ----
xq = r_query_deg * np.cos(np.deg2rad(az_query_deg))
yq = r_query_deg * np.sin(np.deg2rad(az_query_deg))
print(f"Field point: (thx, thy) = ({xq:+.4f}, {yq:+.4f}) deg   "
      f"[r = {r_query_deg} deg, az = {az_query_deg} deg]")
print(f"Convention: s = {s:+d}, spin_sign = {spin_sign:+d};  "
      f"Z(theta) = O + exp(i*spin*s*theta) * C(r, phi - s*theta)\n")

recon = {th: reconstruct(xq, yq, th) for th in rotator_angles_deg}

cols = {"Zernike": [f"Z{j}" for j in js], "j": list(js),
        "spin": [abs(NOLL_NM[j][1]) for j in js],
        "OCS": [recon[rotator_angles_deg[0]][j][0] for j in js],
        "CCS_raw": [recon[rotator_angles_deg[0]][j][1] for j in js]}
for th in rotator_angles_deg:
    cols[f"CCS@{th:+.0f}"] = [recon[th][j][2] for j in js]
    cols[f"total@{th:+.0f}"] = [recon[th][j][3] for j in js]

ref = Table(cols)
for c in ref.colnames:
    if c not in ("Zernike", "j", "spin"):
        ref[c].format = "%+.4f"
if not np.all(np.isfinite(list(cols["OCS"]))):
    print("WARNING: some values are NaN — the field point is outside the sampled disk; "
          "reduce r_query_deg slightly (e.g. 1.65).\n")
ref.pprint(max_width=-1, max_lines=-1)